# Korekcja rolling shutter: dwie kamery (A + B obrócona o 90°)

**Przed startem:** `Środowisko wykonawcze → Zmień typ środowiska → GPU (T4)`.

1. Wgraj cały folder `rs_synth` na swój Google Drive, do `MyDrive/rs_synth`.
2. Uruchamiaj komórki po kolei. Wyniki treningu zapisują się na Drive, więc rozłączenie Colaba ich nie skasuje.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/rs_synth
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Zdjęcia źródłowe: DIV2K (800 ostrych zdjęć 2K, ok. 3,5 GB, kilka minut)
Jeśli masz już gotowy zbiór na Drive (krok 2b), pomiń kroki 1 i 2.

In [ ]:
!wget -q -c http://data.vision.ee.ethz.ch/cvl/DIV2K/DIV2K_train_HR.zip -O /content/div2k.zip
!unzip -q -o /content/div2k.zip -d /content/div2k && ls /content/div2k/DIV2K_train_HR | wc -l

## 2. Generowanie zbioru (800 zdjęć × 20 próbek = 16 tys. par, 256×256)
Na 2 rdzeniach Colaba to ok. 1 h. Na szybki test ustaw `--per-image 2`.
Parametry kamer są w `config_webcam.json`; po pomiarach kolegi zmień tam czas odczytu, odstęp kamer i przesunięcie synchronizacji.

In [ ]:
!python synth_rs.py --src /content/div2k/DIV2K_train_HR --out /content/dataset \
    --per-image 20 --size 256 --workers 2 --config config_webcam.json
!python preview.py /content/dataset/train --n 4 --out /content/preview.png
from IPython.display import Image; Image('/content/preview.png')

### 2b. Kopia zbioru na Drive (żeby nie generować drugi raz)

In [ ]:
# zapis:
!cd /content && zip -qr /content/drive/MyDrive/rs_dataset_256.zip dataset
# odczyt w nowej sesji (zamiast kroków 1-2):
# !unzip -q /content/drive/MyDrive/rs_dataset_256.zip -d /content

## 3. Trening głównego modelu (`motion`): dwie kamery
Sieć porównuje cechy obu kamer (warstwa korelacji), przewiduje 9 parametrów ruchu kamery,
a wektory przesunięć liczy z dokładnej geometrii rolling shutter (jak Velocity-Net u Zhuang i in. 2019).
Checkpointy trafiają do `MyDrive/rs_runs/<nazwa>/`. Po rozłączeniu dodaj `--resume .../last.pt`.

In [ ]:
!python train.py --data /content/dataset --mode motion --epochs 30 --bs 16 \
    --out /content/drive/MyDrive/rs_runs --name motion_dual

## 4. Eksperymenty porównawcze (do pracy)
* `motion_single`: tylko kamera A, odpowiada na pytanie „czy kamera B pomaga?”
* `flow_dual`: U-Net przewidujący gęste pole przesunięć bez modelu geometrycznego (porównanie architektur)

In [ ]:
!python train.py --data /content/dataset --mode motion --single --epochs 30 --bs 16 --out /content/drive/MyDrive/rs_runs --name motion_single
!python train.py --data /content/dataset --mode flow --epochs 30 --bs 16 --out /content/drive/MyDrive/rs_runs --name flow_dual

## 5. Ewaluacja: PSNR, SSIM, czas inferencji (na zbiorze val)

In [ ]:
for name in ['motion_dual', 'motion_single', 'flow_dual']:
    print('=====', name)
    !python train.py --data /content/dataset --eval /content/drive/MyDrive/rs_runs/{name}/best.pt

In [ ]:
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('/content/drive/MyDrive/rs_runs/motion_dual/preview_ep*.png'))[-1:]:
    print(p, '\nkolumny: RS A | RS B (obrócona z powrotem) | wynik | GS cel'); display(Image(p))

# ETAP 2: U-Net doszlifowujący (MotionNet + U-Net)
Wymaga wytrenowanego `rs_runs/motion_dual/best.pt` (krok 3) i **nowej wersji plików** w `MyDrive/rs_synth`.
Zbioru nie trzeba generować od nowa.

MotionNet jest zamrożony; uczy się tylko U-Net, który:
* poprawia lokalnie pola przesunięć dla A i B,
* dopasowuje globalne przesunięcie kamery B (odstęp kamer i synchronizacja),
* decyduje, z której kamery wziąć każdy piksel (brzegi, które wyszły poza kadr A, bierze z B),
* dodaje drobną poprawkę obrazu (ostrość, jasność).

### 6a. Nowa sesja Colaba? Najpierw uruchom komórkę 1 (Drive), potem tę:

In [ ]:
# rozpakowanie zbioru zapisanego w kroku 2b (pomija pliki, które już są)
!unzip -q -n /content/drive/MyDrive/rs_dataset_256.zip -d /content
!ls /content/dataset/train | wc -l

### 6b. Trening (ETAP 2)
Checkpoint zawiera wagi OBU sieci, więc lokalnie wystarczy jeden plik `refine_dual/best.pt`.

In [ ]:
!python train.py --data /content/dataset --mode refine \
    --motion-ckpt /content/drive/MyDrive/rs_runs/motion_dual/best.pt \
    --epochs 15 --bs 16 --out /content/drive/MyDrive/rs_runs --name refine_dual
# po rozłączeniu: dodaj --resume /content/drive/MyDrive/rs_runs/refine_dual/last.pt

### 6c. Porównanie: sam MotionNet vs MotionNet + U-Net
* `psnr` / `ssim`: obszar widoczny w kamerze A (porównywalny z wcześniejszymi wynikami)
* `psnr_b`: brzegi widoczne **tylko** w kamerze B (tu U-Net powinien dać największy zysk)
* `epe`, `epe_b`: średni błąd wektorów przesunięć A i B w pikselach; `epe_in` = bez korekcji

In [ ]:
for name in ['motion_dual', 'refine_dual']:
    print('=====', name)
    !python train.py --data /content/dataset --eval /content/drive/MyDrive/rs_runs/{name}/best.pt

In [ ]:
from IPython.display import Image, display
import glob
p = sorted(glob.glob('/content/drive/MyDrive/rs_runs/refine_dual/preview_ep*.png'))[-1]
print(p, '\nkolumny: RS A | RS B | wynik | GS cel | mapa widoczności (ziel. = A, nieb. = tylko B, czerw. = żadna)')
display(Image(p))

### 6d. Przykładowe wyniki (ten sam skrypt, którego użyjecie lokalnie) + pobranie wag

In [ ]:
!python infer.py --ckpt /content/drive/MyDrive/rs_runs/refine_dual/best.pt \
    --data /content/dataset/val --n 8 --out /content/drive/MyDrive/rs_results
import glob
for p in sorted(glob.glob('/content/drive/MyDrive/rs_results/*.png'))[:4]:
    display(Image(p))

In [ ]:
from google.colab import files
files.download('/content/drive/MyDrive/rs_runs/refine_dual/best.pt')   # ok. 15-20 MB